# FUE Python — Ejemplo: ARMAX con intervenciones

Este notebook muestra el flujo completo:
1. Cargar una serie temporal
2. Especificar un modelo ARMAX con intervenciones
3. Estimar por máxima verosimilitud exacta
4. Diagnosticar los residuos
5. Comparar modelos alternativos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fue import TimeSeries, Intervention, Model

## 1. Serie temporal

Usamos los datos SFNY (Sunspot — New York precipitation proxy), 62 observaciones anuales 1852–1913.

In [ ]:
data = np.array([
    3.91505848, 2.02125792, 0.81208771, 0.60807414, 1.21576447,
    1.43763055, 1.78032601, 0.82841058, 0.65433228, 0.74324607,
    0.93394905, 0.60094494, 0.80840161, 0.90899270, 0.40822203,
    0.41975993, 0.50368768, 0.57248427, 0.72970370, 0.90175445,
    0.61763439, 0.63607641, 0.67670827, 0.81812744, 0.78095914,
    0.82024104, 0.86103433, 0.84442843, 0.74566075, 0.63347579,
    0.72637557, 0.81351610, 0.79142754, 0.80305873, 0.83867533,
    0.98678814, 0.80485863, 0.81651553, 0.75960093, 0.84070968,
    0.89480882, 0.89407591, 0.84323646, 0.77215182, 0.82509544,
    0.87384443, 0.81360106, 0.78497496, 0.71323360, 0.70688522,
    0.81090348, 0.94831097, 0.72598922, 0.80337325, 0.84011493,
    0.89247202, 0.89328246, 0.90942424, 0.82871189, 0.88647340,
    0.82251497, 0.94737336,
])

ts = TimeSeries(data, freq=1, start=(1852, 1), name="SFNY")
print(ts)

In [ ]:
ts.plot()

## 2. ACF / PACF

Exploración de la estructura de autocorrelación.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ts.plot_acf(lags=20, ax=axes[0])
ts.plot_pacf(lags=20, ax=axes[1])
plt.tight_layout()
plt.show()

## 3. Modelo AR(1) simple

Primer modelo de referencia.

In [ ]:
m_ar1 = Model(ts, ar=[[0.5]])
m_ar1.fit()
print(m_ar1.summary())

## 4. Modelo ARMAX con step e intervención de nivel

El modelo SFNY.2 incluye:
- Intervención `step` con función de transferencia ω(B)/δ(B) en 1853
- Dos factores AR regulares: AR(1) × AR(2)
- Media estimada μ
- Transformación logarítmica (boxlam=0)

In [ ]:
step = Intervention(
    "step", at=1,            # step desde la segunda observación (1853)
    omega=[0.08],            # initial ω₀
    omega_free=[True],
    delta=[0.6],             # initial δ₁ (decay)
    delta_free=[True],
)

m_full = Model(
    ts,
    interventions=[step],
    ar=[[0.8], [-0.1, -0.1]],   # AR(1) × AR(2)
    boxlam=0.0,                  # log-transform
    mu=0.0, estimate_mu=True,
)
m_full.fit()
print(m_full.summary())

## 5. Diagnóstico de residuos

In [ ]:
m_full.plot_residuals(lags=20)

## 6. Comparación de modelos

AR(1) vs. modelo ARMAX completo.

In [ ]:
m_ar1.compare(m_full)

## 7. Desde pandas

`TimeSeries.from_pandas()` infiere la frecuencia y la fecha inicial automáticamente.

In [ ]:
try:
    import pandas as pd
    s = pd.Series(data, index=pd.period_range("1852", periods=62, freq="A"))
    ts_pd = TimeSeries.from_pandas(s, name="SFNY_pd")
    print(ts_pd)
    m_pd = Model(ts_pd, ar=[[0.5]])
    m_pd.fit()
    print(f"loglik from pandas Series: {m_pd.loglik:.6f}")
except ImportError:
    print("pandas not installed — omitiendo demo")

## 8. Modelo mensual con componentes estacionales

Ejemplo con intervenciones `cos`/`sin`/`alter` y coeficiente AR fijo.
Datos: RIPC mensual (enero 2002 – diciembre 2007, 72 observaciones).

In [ ]:
ripc_data = np.array([
    0.413459, 0.416226, 0.418544, 0.422442, 0.424508, 0.425892,
    0.425137, 0.425577, 0.427322, 0.429367, 0.432350, 0.434795,
    0.443644, 0.443617, 0.443454, 0.448741, 0.450270, 0.448844,
    0.448505, 0.447079, 0.449154, 0.449650, 0.452374, 0.452679,
    0.452326, 0.452981, 0.453226, 0.454730, 0.449935, 0.447131,
    0.445082, 0.444966, 0.445047, 0.443958, 0.445585, 0.446936,
    0.447090, 0.445735, 0.443441, 0.444167, 0.445403, 0.445490,
    0.442748, 0.439849, 0.437653, 0.438303, 0.442595, 0.445719,
    0.444463, 0.446707, 0.447143, 0.443674, 0.440874, 0.438993,
    0.437829, 0.437908, 0.442587, 0.446552, 0.447956, 0.447148,
    0.447111, 0.445032, 0.441439, 0.438548, 0.436016, 0.436859,
    0.438797, 0.439924, 0.441830, 0.441481, 0.441057, 0.443876,
])

ts_ripc = TimeSeries(ripc_data, freq=12, start=(2002, 1), name="RIPC")

# Intervenciones estacionales: armónicos 1-5 + alternator
seasonal = [
    Intervention(t, harmonic=k, omega=[0.0], omega_free=[True])
    for t, k in [("cos", 1), ("sin", 1), ("cos", 2), ("sin", 2),
                 ("cos", 3), ("sin", 3), ("cos", 4), ("sin", 4),
                 ("cos", 5), ("sin", 5)]
]
seasonal.append(Intervention("alter", omega=[0.0], omega_free=[True]))
seasonal.append(Intervention("step", at=1,
                              omega=[0.014], omega_free=[True],
                              delta=[0.6],   delta_free=[True]))

m_ripc = Model(
    ts_ripc,
    interventions=seasonal,
    ar=[[0.0]], ar_free=[[False]],   # AR(1) fijo en 0
    boxlam=0.0, refactor=100.0,
    mu=0.0, estimate_mu=True,
)
m_ripc.fit()
print(m_ripc.summary())

In [ ]:
m_ripc.plot_residuals(lags=24)